# Capítulo 5 — Serving resiliente: API sem estado, lote, avaliação contínua

## Missão
Capítulo 5, padrões 16 a 20: pôr o modelo para trabalhar e **mantê-lo** confiável. Você vai construir:
- uma **função de serving sem estado** (API Flask com o pipeline inteiro);
- o **serving em lote** para o painel diário;
- a **avaliação contínua**, que mede o erro mês a mês e dispara o retreino quando o mundo muda;
- a **previsão em duas fases**: um modelo pequeno no celular e um grande no servidor;
- as **previsões com chave**, que voltam com o id e não dependem da ordem.

**No fim deste caderno você terá:** o módulo `meu_mlops/servico.py` (API stateless com chave, lote em partes e monitor de drift com gatilho) e o retreino disparado sozinho quando o trânsito piora.

**Leitura essencial (opcional, livro em inglês):**
- Design Pattern 16: Stateless Serving Function · PDF p. 244
- Design Pattern 17: Batch Serving · PDF p. 258
- Design Pattern 18: Continued Model Evaluation · PDF p. 265
- Design Pattern 19: Two-Phase Predictions · PDF p. 280
- Design Pattern 20: Keyed Predictions · PDF p. 293

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap05_serving_resiliente_api_sem_estado_lote_avalia.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_mlops/
os.makedirs("../meu_mlops", exist_ok=True)
open("../meu_mlops/__init__.py", "a").close()        # meu_mlops/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_mlops")

## Ideia em 30 segundos
- **Sem estado:** a função de serving recebe tudo o que precisa na requisição e não guarda nada entre chamadas. Por isso dá para ter 1 ou 100 cópias atrás de um balanceador.
- **Com chave:** a resposta carrega o id de quem perguntou. Num lote processado em paralelo, a ordem se perde, e a chave não.

### 5.1 **O serviço.** Crie `servico.py` com:
- `criar_app(modelo, colunas)`: um Flask com `POST /prever`, que recebe uma **lista** de pedidos (cada um com `pedido_id` e as colunas), devolve `[{"pedido_id": ..., "minutos": ...}]` e recusa com 400 quando falta coluna;
- `prever_em_lote(modelo, df, colunas, tamanho=20000)`: processa em partes e devolve um DataFrame com a chave;
- `Monitor(limite_mae)`: `registrar(mes, y, p)` guarda o MAE e o viés do mês; `precisa_retreinar()` diz se os dois últimos meses passaram do limite.

_A célula grava `meu_mlops/servico.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_mlops/servico.py


### 5.2 **A API sem estado e com chave.** Treine um `HistGradientBoostingRegressor` nas numéricas, crie o app e, pelo `test_client`, mande 3 pedidos **fora de ordem**. Confira que cada resposta volta com o seu `pedido_id`. Mande também um pedido sem `distancia_km`: deve voltar 400.

### 5.3 **Serving em lote.** Pontue os 20 mil pedidos do teste **embaralhados** com o `prever_em_lote` (partes de 5 mil) e junte de volta pela chave. Confira que o MAE é o mesmo da previsão direta. Meça o tempo por pedido do lote e de 200 chamadas individuais à API.

### 5.4 **Avaliação contínua com gatilho.** Gere os meses de julho a dezembro com o trânsito piorando aos poucos (`drift` de 0 a 1,5). Registre o MAE e o viés de cada mês num `Monitor(limite_mae=4.5)`. Quando o gatilho disparar, **retreine** com os últimos 2 meses e siga monitorando com o modelo novo. Mostre a tabela.

### 5.5 **Duas fases.** No celular cabe só um modelo minúsculo (`DecisionTreeRegressor(max_depth=3)`) que responde na hora. Se a previsão dele passar de 40 min, a pergunta vai para o modelo grande no servidor. Compare o MAE e a fração de pedidos que precisaram ir ao servidor contra "tudo no modelo grande" e "tudo no pequeno".

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
